# 03 · DIN Attention Visualisation

Loads the saved DIN model and shows, for 5 random users, how attention
redistributes when the **same user history** is scored against their
actual held-out positive.

Image artefact: `experiments/results/ranking/din_attention_heatmap.png`
(also embedded into README §7.2).

## 1 · Setup

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path('.').resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from hydra import compose, initialize_config_dir
from neorec.ranking.din import DINRanker
from neorec.ranking.features import RankingFeaturizer

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': False})

## 2 · Re-instantiate trained DIN

In [ ]:
with initialize_config_dir(version_base='1.3', config_dir=str(ROOT / 'configs')):
    cfg = compose(config_name='config', overrides=['rank=din'])

PROC = ROOT / 'data' / 'processed' / 'movielens_1m'
featurizer = RankingFeaturizer(processed_dir=PROC, max_genres=6, max_seq_len=50)
train_df = (pd.read_parquet(PROC / 'interactions.parquet')
            .merge(pd.read_parquet(PROC / 'split.parquet')[['user_id','item_id','split']],
                   on=['user_id','item_id'], how='inner')
            .query('split == "train"')
            .reset_index(drop=True))
featurizer.build_sequences(train_df)
ranker = DINRanker(cfg, featurizer)
ranker.load(ROOT / 'artifacts' / 'rank' / 'din')
print('DIN ready, embedding_dim =', ranker.model.embedding_dim,
      'use_attention =', ranker.model.use_attention)

## 3 · Pick 5 users + their held-out positives

We filter to users whose history fills the full `max_seq_len=50` —
the heatmap is more interpretable when no padding rows show through.

In [ ]:
test_df = (pd.read_parquet(PROC / 'split.parquet')
           .query('split == "test"').reset_index(drop=True))
items = pd.read_parquet(PROC / 'item_features.parquet').set_index('item_id')
titles = items['title'].to_dict()

rng = np.random.default_rng(42)
hist_len = featurizer._user_history_mask.sum(axis=1)
rich = np.where(hist_len >= featurizer.max_seq_len)[0]
test_users = set(test_df['user_id'].tolist())
users = [u for u in rich if u in test_users]
rng.shuffle(users)
users = users[:5]
test_idx = test_df.set_index('user_id')['item_id'].to_dict()
targets = [test_idx[u] for u in users]
print('users:', users)
print('targets:', [titles.get(t,'?')[:40] for t in targets])

## 4 · Compute attention weights and visualise

In [ ]:
w, history, mask = ranker.attention_for_users(
    np.array(users, dtype=np.int64), np.array(targets, dtype=np.int64))

w_norm = np.zeros_like(w, dtype=np.float32)
for i in range(w.shape[0]):
    valid = w[i][mask[i] > 0]
    if valid.size == 0: continue
    a, b = float(valid.min()), float(valid.max())
    w_norm[i] = (w[i] - a) / max(b - a, 1e-6)
    w_norm[i][mask[i] == 0] = np.nan

fig, ax = plt.subplots(figsize=(14, 5.5))
im = ax.imshow(w_norm, cmap='magma', aspect='auto', vmin=0., vmax=1.)
ax.set_yticks(range(len(users)))
ax.set_yticklabels([f'u{u} → ' + titles.get(int(t), f'i{t}')[:35]
                    for u, t in zip(users, targets)], fontsize=10)
ax.set_xlabel('History position (older → newer)')
ax.set_title('DIN attention weights · target item vs user history')
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label='relative attention')
plt.tight_layout()
plt.show()

## 5 · Discussion

* **Bright cells** are history items the attention unit considers most
  similar to the target. In the well-behaved cases, you'll see a few
  ‘spikes’ aligned with movies of the same genre or franchise as the target.
* **Even rows** indicate attention failed to discriminate — the model
  fell back to sum-pooling.  This happens when the user's history is too
  homogeneous (e.g. a casual viewer who rates only blockbusters).
* The ablation result in §7.2 shows: full DIN gets +3.2 pts AUC over the
  no-attention baseline, but its end-to-end Recall@10 drops because the
  CTR head over-trusts history-target similarity on hard negatives
  sampled by the recall layer.  This motivates **hard-negative mining**
  in W6.